# Cross-task patching — aggregation-head OV only
`experiments/interplay/` · transplant at the **final/query position**

Aggregation heads attend **final position → output positions** and write their result
**at the final position**. So patching *only their OV* there installs the donor prompt's
**integrated** write — the most task-vector-like object in the circuit — and nothing else.
Single position ⇒ no positional alignment needed.

- **acc_donor rises** → the aggregation write *is* a transplantable task code (even for nonce,
  this would mean the integrated representation behaves like a TV).
- **acc_donor flat, acc_src → other** → integration is necessary but prompt-specific, not transplantable.
- **acc_leak rises** → the write carries the donor's *answer content*, not a reusable mapping
  (task-recognition reading).

Bins (exact full-string match, priority order):
`acc_src` = source_rule(src_input) · `acc_donor` = donor_rule(src_input) ·
`acc_leak` = donor prompt's stored answer · `other` = none.

Full 10×10 source×donor matrices per family (diagonal = self-transplant control; on it
donor_rule==source_rule, so it scores as `acc_src` — should stay high if the patch is non-destructive).


In [1]:
import itertools, pickle, random
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
import sys
REPO = Path('../..').resolve()            # experiments/interplay -> repo root
sys.path.insert(0, str(REPO))

# ---- ADAPT: repo imports -------------------------------------------------
from data.tasks import apply_rule
from utils.eval import check_correct_multitoken
from utils.extraction import load_model
from utils.heads import load_aggregation_heads

SPLITS = REPO/'data/nonce_arithmetic_splits.pkl'
OUTDIR = REPO/'results/interplay'; OUTDIR.mkdir(parents=True, exist_ok=True)
N       = 10
MAX_NEW = 8
SEED    = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model, tok, dev = load_model()
d_head = model.config.hidden_size // model.config.num_attention_heads

with open(SPLITS,'rb') as f: splits = pickle.load(f)
aggregation = load_aggregation_heads()
print('aggregation heads:', sorted(aggregation))

# ---- ADAPT: family membership -------------------------------------------
def fam(t): return splits[t].get('family', 'arith' if t[0].isdigit() else 'nonce')
nonce = [t for t in splits if fam(t)=='nonce']
arith = [t for t in splits if fam(t)=='arith']
print('nonce', len(nonce), 'arith', len(arith))

ImportError: cannot import name 'apply_rule' from 'data.tasks' (/workspace/TAU/data/tasks.py)

In [ ]:
def final_position(prompt):
    return len(tok(prompt, add_special_tokens=True)['input_ids']) - 1

def heads_by_layer(heads):
    d=defaultdict(list)
    for L,h in heads: d[L].append(h)
    return dict(d)
agg_hbl = heads_by_layer(aggregation)

class OVCapture:               # o_proj input slice (= head z) at one position
    def __init__(self, hbl, pos): self.hbl,self.pos,self.store,self.hs=hbl,pos,{},[]
    def __enter__(self):
        def mk(L):
            def hook(m,args,kw=None):
                x=args[0]
                for h in self.hbl[L]:
                    s=slice(h*d_head,(h+1)*d_head)
                    self.store[(L,h)]=x[0,self.pos,s].detach().clone()
                return None
            return hook
        for L in self.hbl:
            self.hs.append(model.model.layers[L].self_attn.o_proj
                           .register_forward_pre_hook(mk(L)))
        return self
    def __exit__(self,*a):
        for h in self.hs: h.remove()

class OVPatch:                 # overwrite head z at one position, prefill only
    def __init__(self, hbl, pos, donor): self.hbl,self.pos,self.donor,self.hs=hbl,pos,donor,[]
    def __enter__(self):
        def mk(L):
            def hook(m,args,kw=None):
                x=args[0]
                if x.shape[1]==1: return None
                x=x.clone()
                for h in self.hbl[L]:
                    s=slice(h*d_head,(h+1)*d_head)
                    x[0,self.pos,s]=self.donor[(L,h)].to(x.dtype)
                return (x,)
            return hook
        for L in self.hbl:
            self.hs.append(model.model.layers[L].self_attn.o_proj
                           .register_forward_pre_hook(mk(L)))
        return self
    def __exit__(self,*a):
        for h in self.hs: h.remove()

@torch.no_grad()
def capture_agg(prompt):
    enc=tok(prompt, return_tensors='pt').to(dev)
    cap=OVCapture(agg_hbl, [final_position(prompt)])
    with cap: model(**enc)
    return cap.store

@torch.no_grad()
def patched_gen(prompt, donor_store):
    enc=tok(prompt, return_tensors='pt').to(dev)
    with OVPatch(agg_hbl, [final_position(prompt)], donor_store):
        gen=model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(gen[0, enc.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
rows=[]
for family, tasks in [('nonce',nonce),('arith',arith)]:
    dcache={t:[capture_agg(p['prompt']) for p in splits[t]['icl_prompts'][:N]]
            for t in tasks}                              # donor agg-OV per task
    for src_t in tqdm(tasks, desc=family):
        src_ps=splits[src_t]['icl_prompts'][:N]
        for dnr_t in tasks:                              # full matrix incl diagonal
            dnr_ps=splits[dnr_t]['icl_prompts'][:N]
            c=dict(acc_src=0,acc_donor=0,acc_leak=0,other=0)
            for j,sp in enumerate(src_ps):
                dec=patched_gen(sp['prompt'], dcache[dnr_t][j])
                s_tgt=apply_rule(src_t, sp['query_input'])
                d_tgt=apply_rule(dnr_t, sp['query_input'])
                l_tgt=dnr_ps[j]['query_output']
                if   check_correct_multitoken(dec,s_tgt): c['acc_src']+=1
                elif check_correct_multitoken(dec,d_tgt): c['acc_donor']+=1
                elif check_correct_multitoken(dec,l_tgt): c['acc_leak']+=1
                else: c['other']+=1
            rows.append(dict(family=family,src=src_t,donor=dnr_t,
                             **{k:v/N for k,v in c.items()}))
R=pd.DataFrame(rows)
R.to_csv(OUTDIR/'08_crosstask_agg_ov.csv', index=False)
print(R.groupby('family')[['acc_src','acc_donor','acc_leak','other']].mean().round(3))

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns

BINS=['acc_donor','acc_src','acc_leak','other']
TITLES={'acc_donor':'transplanted (donor) task','acc_src':'original (source) task',
        'acc_leak':'donor answer leaked','other':'broken / other'}
for family in ['nonce','arith']:
    sub=R[R.family==family]
    order=sorted(sub['src'].unique())
    fig,axes=plt.subplots(1,4,figsize=(22,5))
    for ax,b in zip(axes,BINS):
        mat=sub.pivot(index='src',columns='donor',values=b).reindex(index=order,columns=order)
        sns.heatmap(mat,ax=ax,vmin=0,vmax=1,cmap='viridis',annot=True,fmt='.2f',
                    annot_kws={'size':7},cbar_kws={'shrink':.7})
        ax.set(title=f'{TITLES[b]}',xlabel='donor',ylabel='source')
    fig.suptitle(f'{family}: aggregation-head OV transplant (final position), N={N}',y=1.02)
    plt.tight_layout()
    plt.savefig(OUTDIR/f'08_crosstask_agg_ov_{family}.png',dpi=200,bbox_inches='tight')
    plt.show()
print('saved heatmaps')

## Read it
Rows = **source** prompt, columns = **donor** (transplanted) task.

- **`transplanted (donor)` heatmap bright off-diagonal** → aggregation-head OV carries a
  transplantable task code; the integrated write installs the donor task on the source query.
- **`original (source)` bright on the diagonal but dark off-diagonal** → the patch reliably
  *replaces* the source task (necessary), and the diagonal confirms a same-task donor is non-destructive.
- **`donor answer leaked` bright** → write carries answer content, not a mapping (recognition, not learning).
- **nonce vs arith**: compare the donor heatmaps. Transfer in one family but not the other = the
  mechanism boundary. Block structure within a heatmap (clusters of tasks that transfer to each
  other) = task-similarity structure in the aggregated code.

Sanity: spot-check one decoded `dec` vs an unpatched generation, and confirm the diagonal of the
`original` heatmap is high before trusting off-diagonal nulls.